In [1]:
import warnings
import joblib
import pandas as pd
import numpy as np
import keras
from keras import layers
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

warnings.filterwarnings('ignore')

In [2]:
train_df = pd.read_csv('archive-4/train.csv', on_bad_lines='skip')
val_df = pd.read_csv('archive-4/val.csv', on_bad_lines='skip')
test_df = pd.read_csv('archive-4/test.csv', on_bad_lines='skip')

In [3]:
df_train = pd.concat([train_df, val_df], ignore_index=True)
df_test = test_df.copy()

In [9]:
for df in [df_train, df_test]:
    if 'User_ID' in df.columns:
        df.drop(columns=['User_ID'], inplace=True)

target_col = 'Dominant_Emotion'
num_cols = ['Age', 'Daily_Usage_Time (minutes)', 'Posts_Per_Day', 
            'Likes_Received_Per_Day', 'Comments_Received_Per_Day', 'Messages_Sent_Per_Day']

In [10]:
for col in num_cols:
    df_train[col] = pd.to_numeric(df_train[col], errors='coerce')
df_train = df_train.dropna()

X_train = df_train.drop(columns=[target_col])
y_train = df_train[target_col]

In [11]:
for col in num_cols:
    df_test[col] = pd.to_numeric(df_test[col], errors='coerce')
df_test = df_test.dropna()

X_test = df_test.drop(columns=[target_col])
y_test = df_test[target_col]

In [12]:
for col in ['Gender', 'Platform']:
    le = LabelEncoder()
    combined = pd.concat([X_train[col], X_test[col]]).astype(str)
    le.fit(combined)
    X_train[col] = le.transform(X_train[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

In [13]:
target_encoder = LabelEncoder()
y_train_encoded = target_encoder.fit_transform(y_train)
y_test_encoded = target_encoder.transform(y_test)
num_classes = len(target_encoder.classes_)

In [14]:
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

In [15]:
y_train_cat = keras.utils.to_categorical(y_train_encoded, num_classes=num_classes)
y_test_cat = keras.utils.to_categorical(y_test_encoded, num_classes=num_classes)

In [16]:
input_dim = X_train.shape[1]
model = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

print('\n--- Training Neural Network ---')
model.fit(
    X_train, y_train_cat,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)


--- Training Neural Network ---
Epoch 1/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.2263 - loss: 2.2517 - val_accuracy: 0.3396 - val_loss: 1.8119
Epoch 2/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.3898 - loss: 1.7217 - val_accuracy: 0.4575 - val_loss: 1.6472
Epoch 3/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4562 - loss: 1.5223 - val_accuracy: 0.4717 - val_loss: 1.5587
Epoch 4/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4763 - loss: 1.4731 - val_accuracy: 0.4764 - val_loss: 1.4949
Epoch 5/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5118 - loss: 1.3964 - val_accuracy: 0.4764 - val_loss: 1.4369
Epoch 6/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4976 - loss: 1.3689 - val_accuracy: 0.4811 - val_loss: 1.3891
Epoch 7/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5332 - loss: 1.3454 - val_accuracy: 0.5047 - val_loss: 1.3509
Epoch 8/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5213 - loss: 1.3020 -

In [17]:
y_prob = model.predict(X_test)
y_pred = np.argmax(y_prob, axis=1)

acc = accuracy_score(y_test_encoded, y_pred)
f1 = f1_score(y_test_encoded, y_pred, average='weighted')

print(f"\n>> Deep Learning Model Test Accuracy: {acc:.4f} | Weighted F1-Score: {f1:.4f}")

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step

>> Deep Learning Model Test Accuracy: 0.8627 | Weighted F1-Score: 0.8627


In [18]:
model.save('best_dl_model.keras')
joblib.dump(scaler, 'scaler_dl.pkl')
joblib.dump(target_encoder, 'target_encoder_dl.pkl')

['target_encoder_dl.pkl']